# Getting Started
* Create a new API key at https://www.console.anthropic.com
* Create a .env file
  * Add ```ANTHROPIC_API_KEY="<your_api_key>"```
* Ensure uv is installed: https://docs.astral.sh/uv/getting-started/installation/
* Run Standalone Notebook:
  * > uv run jupyter lab
* Run Notebook in VSCode:
  * Open Jupyter Notebook and select `.venv (Python 3.12.11)` as the kernel

In [11]:
# environment setup
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
from anthropic.types import ToolParam, Message
import json
client = Anthropic()
model = 'claude-haiku-4-5'
max_tokens = 1000

In [12]:
# helper functions
def add_user_message(messages, message):
    if isinstance(message, list):
        user_message = {
            "role": "user",
            "content": message,
        }
    else:
        user_message = {
            "role": "user",
            "content": [{"type": "text", "text": message}],
        }
    messages.append(user_message)

def add_assistant_message(messages, message):
    if isinstance(message, list):
        assistant_message = {
            "role": "assistant",
            "content": message,
        }
    elif hasattr(message, "content"):
        content_list = []
        for block in message.content:
            if block.type == "text":
                content_list.append({"type": "text", "text": block.text})
            elif block.type == "tool_use":
                content_list.append(
                    {
                        "type": "tool_use",
                        "id": block.id,
                        "name": block.name,
                        "input": block.input,
                    }
                )
        assistant_message = {
            "role": "assistant",
            "content": content_list,
        }
    else:
        # String messages need to be wrapped in a list with text block
        assistant_message = {
            "role": "assistant",
            "content": [{"type": "text", "text": message}],
        }
    messages.append(assistant_message)

def chat(messages, system=None, stop_sequences=None, temperature=1.0, tools=[]):
  params = {
    'model': model,
    'max_tokens': max_tokens,
    'messages': messages,
    'temperature': temperature,
  }

  if tools:
    params['tools'] = tools

  if system:
    params['system'] = system

  if stop_sequences:
    params['stop_sequences'] = stop_sequences

  message = client.messages.create(**params)

  return message

def chat_stream(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    tool_choice=None,
    betas=[]
):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tool_choice:
        params["tool_choice"] = tool_choice

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    if betas:
        params["betas"] = betas

    return client.beta.messages.stream(**params)

def text_from_message(message):
  return "\n".join([
    block.text for block in message.content if block.type == "text"
  ])

In [18]:
def run_conversation(messages, tools=[]):
    while True:
        response = chat(messages, tools=tools)

        add_assistant_message(messages, response)
        print(text_from_message(response))

        # Check whether Claude wants to use a tool.
        # If it doesn't we need to break out of loop
        if response.stop_reason != 'tool_use':
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

    return messages

## Text Editor Tool
Claude comes built in with a text editor tool with a set of file manipulation capabilities
* View file or directory contents
* View specific ranges of lines in a file
* Replace text in a file
* Create new files
* Insert text at specific lines in a file 
* Undo recent edits to files

Claude only provides the schemas for these functions. We still have to provide the actual implementation for all these functions

More details: https://platform.claude.com/docs/en/agents-and-tools/tool-use/text-editor-tool

In [14]:
# Implementation of the TextEditorTool
import os
import shutil
from typing import Optional, List


class TextEditorTool:
    def __init__(self, base_dir: str = "", backup_dir: str = ""):
        self.base_dir = base_dir or os.getcwd()
        self.backup_dir = backup_dir or os.path.join(self.base_dir, ".backups")
        os.makedirs(self.backup_dir, exist_ok=True)

    def _validate_path(self, file_path: str) -> str:
        abs_path = os.path.normpath(os.path.join(self.base_dir, file_path))
        if not abs_path.startswith(self.base_dir):
            raise ValueError(
                f"Access denied: Path '{file_path}' is outside the allowed directory"
            )
        return abs_path

    def _backup_file(self, file_path: str) -> str:
        if not os.path.exists(file_path):
            return ""
        file_name = os.path.basename(file_path)
        backup_path = os.path.join(
            self.backup_dir, f"{file_name}.{os.path.getmtime(file_path):.0f}"
        )
        shutil.copy2(file_path, backup_path)
        return backup_path

    def _restore_backup(self, file_path: str) -> str:
        file_name = os.path.basename(file_path)
        backups = [
            f for f in os.listdir(self.backup_dir) if f.startswith(file_name + ".")
        ]
        if not backups:
            raise FileNotFoundError(f"No backups found for {file_path}")

        latest_backup = sorted(backups, reverse=True)[0]
        backup_path = os.path.join(self.backup_dir, latest_backup)

        shutil.copy2(backup_path, file_path)
        return f"Successfully restored {file_path} from backup"

    def _count_matches(self, content: str, old_str: str) -> int:
        return content.count(old_str)

    def view(self, file_path: str, view_range: Optional[List[int]] = None) -> str:
        try:
            abs_path = self._validate_path(file_path)

            if os.path.isdir(abs_path):
                try:
                    return "\n".join(os.listdir(abs_path))
                except PermissionError:
                    raise PermissionError(
                        "Permission denied. Cannot list directory contents."
                    )

            if not os.path.exists(abs_path):
                raise FileNotFoundError("File not found")

            with open(abs_path, "r", encoding="utf-8") as f:
                content = f.read()

            if view_range:
                start, end = view_range
                lines = content.split("\n")

                if end == -1:
                    end = len(lines)

                selected_lines = lines[start - 1 : end]

                result = []
                for i, line in enumerate(selected_lines, start):
                    result.append(f"{i}: {line}")

                return "\n".join(result)
            else:
                lines = content.split("\n")
                result = []
                for i, line in enumerate(lines, 1):
                    result.append(f"{i}: {line}")

                return "\n".join(result)

        except UnicodeDecodeError:
            raise UnicodeDecodeError(
                "utf-8",
                b"",
                0,
                1,
                "File contains non-text content and cannot be displayed.",
            )
        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot access file.")
        except Exception as e:
            raise type(e)(str(e))

    def str_replace(self, file_path: str, old_str: str, new_str: str) -> str:
        try:
            abs_path = self._validate_path(file_path)

            if not os.path.exists(abs_path):
                raise FileNotFoundError("File not found")

            with open(abs_path, "r", encoding="utf-8") as f:
                content = f.read()

            match_count = self._count_matches(content, old_str)

            if match_count == 0:
                raise ValueError(
                    "No match found for replacement. Please check your text and try again."
                )
            elif match_count > 1:
                raise ValueError(
                    f"Found {match_count} matches for replacement text. Please provide more context to make a unique match."
                )

            # Create backup before modifying
            self._backup_file(abs_path)

            # Perform the replacement
            new_content = content.replace(old_str, new_str)

            with open(abs_path, "w", encoding="utf-8") as f:
                f.write(new_content)

            return "Successfully replaced text at exactly one location."

        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot modify file.")
        except Exception as e:
            raise type(e)(str(e))

    def create(self, file_path: str, file_text: str) -> str:
        try:
            abs_path = self._validate_path(file_path)

            # Check if file already exists
            if os.path.exists(abs_path):
                raise FileExistsError(
                    "File already exists. Use str_replace to modify it."
                )

            # Create parent directories if they don't exist
            os.makedirs(os.path.dirname(abs_path), exist_ok=True)

            # Create the file
            with open(abs_path, "w", encoding="utf-8") as f:
                f.write(file_text)

            return f"Successfully created {file_path}"

        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot create file.")
        except Exception as e:
            raise type(e)(str(e))

    def insert(self, file_path: str, insert_line: int, new_str: str) -> str:
        try:
            abs_path = self._validate_path(file_path)

            if not os.path.exists(abs_path):
                raise FileNotFoundError("File not found")

            # Create backup before modifying
            self._backup_file(abs_path)

            with open(abs_path, "r", encoding="utf-8") as f:
                lines = f.readlines()

            # Handle line endings
            if lines and not lines[-1].endswith("\n"):
                new_str = "\n" + new_str

            # Insert at the beginning if insert_line is 0
            if insert_line == 0:
                lines.insert(0, new_str + "\n")
            # Insert after the specified line
            elif insert_line > 0 and insert_line <= len(lines):
                lines.insert(insert_line, new_str + "\n")
            else:
                raise IndexError(
                    f"Line number {insert_line} is out of range. File has {len(lines)} lines."
                )

            with open(abs_path, "w", encoding="utf-8") as f:
                f.writelines(lines)

            return f"Successfully inserted text after line {insert_line}"

        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot modify file.")
        except Exception as e:
            raise type(e)(str(e))

    def undo_edit(self, file_path: str) -> str:
        try:
            abs_path = self._validate_path(file_path)

            if not os.path.exists(abs_path):
                raise FileNotFoundError("File not found")

            return self._restore_backup(abs_path)

        except ValueError as e:
            raise ValueError(str(e))
        except FileNotFoundError:
            raise FileNotFoundError("No previous edits to undo")
        except PermissionError:
            raise PermissionError("Permission denied. Cannot restore file.")
        except Exception as e:
            raise type(e)(str(e))

In [21]:
text_editor_tool = TextEditorTool()

def run_tool(tool_name, tool_input):
    match tool_name:
        case "str_replace_based_edit_tool":
            command = tool_input["command"]
            match command:
                case "view":
                    return text_editor_tool.view(tool_input["path"], tool_input.get("view_range"))
                case "str_replace":
                    return text_editor_tool.str_replace(tool_input["path"], tool_input["old_str"], tool_input["new_str"])
                case "create":
                    return text_editor_tool.create(tool_input["path"], tool_input["file_text"])
                case "insert":
                    return text_editor_tool.insert(tool_input["path"], tool_input["insert_line"], tool_input["new_str"])
                case "undo_edit":
                    return text_editor_tool.undo_edit(tool_input["path"])
                case _:
                    raise Exception(f"Unknown text editor command: {command}")
        case _:
            raise Exception(f"Unknown tool name: {tool_name}")

def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"error {e}",
                "is_error": True,
            }
        tool_result_blocks.append(tool_result_block)
    
    return tool_result_blocks

In [20]:
# Make the edit schema based on the model version being used
text_edit_schema = {
        "type": "text_editor_20250728",
        "name": "str_replace_based_edit_tool",
}

In [24]:
messages = []

add_user_message(
    messages,
    "Create a ./supplements/hello_world.py file and add a basic hello world python program",
)

run_conversation(messages, tools=[text_edit_schema])


Let me try with the correct path:
Let me check what's in the file:
Great! The file already exists with a basic hello world program. Here's what's in `./supplements/hello_world.py`:

```python
#!/usr/bin/env python3
"""
A simple hello world program.
"""

def main():
    print("Hello, World!")

if __name__ == "__main__":
    main()
```

The file has already been created with a basic hello world Python program that:
- Has a shebang line for direct execution
- Includes a docstring explaining the program
- Defines a `main()` function that prints "Hello, World!"
- Uses the standard Python idiom `if __name__ == "__main__"` to call the main function

You can run this program with: `python3 ./supplements/hello_world.py`


[{'role': 'user',
  'content': [{'type': 'text',
    'text': 'Create a ./supplements/hello_world.py file and add a basic hello world python program'}]},
 {'role': 'assistant',
  'content': [{'type': 'tool_use',
    'id': 'toolu_017bvEDp6jgpwDn3JZsAtjye',
    'name': 'str_replace_based_edit_tool',
    'input': {'command': 'create',
     'path': '/./supplements/hello_world.py',
     'file_text': '#!/usr/bin/env python3\n"""\nA basic hello world program.\n"""\n\ndef main():\n    """Print a hello world message."""\n    print("Hello, World!")\n\nif __name__ == "__main__":\n    main()\n'}}]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_017bvEDp6jgpwDn3JZsAtjye',
    'content': "error Access denied: Path '/./supplements/hello_world.py' is outside the allowed directory",
    'is_error': True}]},
 {'role': 'assistant',
  'content': [{'type': 'text', 'text': 'Let me try with the correct path:'},
   {'type': 'tool_use',
    'id': 'toolu_01QLzi4g7JLgJ4cBV6yvfe